# YouTube Transcript RAG Chatbot
End-to-end notebook: fetch transcripts, chunk, embed, store in Chroma, answer questions only from context.

# Step 1: Select topic\n\nTopic: Gen-AI / LLM educational videos. Swap `VIDEO_IDS` below for any other topic/videos you like — the rest of the pipeline is unchanged.

In [ ]:
VIDEO_IDS = [
    "zjkBMFhNj_g",  # Andrej Karpathy - "Intro to Large Language Models"
]
VIDEO_IDS

# Step 2: Fetch transcripts and store as text files\n\nUses the YouTube Transcript API (captions only, no OAuth needed). Saves each transcript to `transcripts/<video_id>.txt`, skipping videos already fetched.

In [ ]:
from rag import fetch_transcripts

available_video_ids = fetch_transcripts(VIDEO_IDS)
available_video_ids

# Step 3: Split transcripts into chunks\n\nUses `RecursiveCharacterTextSplitter`. Each chunk's metadata records its `video_id` and `chunk_index`, which is used as a "page no." style reference since transcripts have no real page numbers.

In [ ]:
from rag import load_and_split_documents

documents = load_and_split_documents(available_video_ids)
print(f"{len(documents)} chunks")
documents[0]

# Step 4: Generate embeddings and store in Chroma\n\nEmbeddings model and LLM are both configured in `model_config.py` — change `EMBEDDING_PROVIDER`/`EMBEDDING_MODEL`/`LLM_PROVIDER`/`LLM_MODEL` there (or via a `.env` file) to swap models without touching this notebook. Default: `nomic-embed-text` via local Ollama, persisted under `chroma_db/`.

In [ ]:
from rag import get_or_build_vectorstore

vectorstore = get_or_build_vectorstore(available_video_ids)
vectorstore

# Step 5 & 6: Build the QA chain and ask a question\n\nThe chain retrieves the top-k relevant chunks, then asks the configured LLM to answer *only* from that context (including the chunk source as a page-no.-style citation). If the context doesn't contain the answer, it responds "I don't know."

In [ ]:
from rag import answer_question, build_qa_chain

qa_chain = build_qa_chain(vectorstore)

result = answer_question(qa_chain, "What is this video about?")
print(result["answer"])
print("Sources:", result["sources"])

In [ ]:
# Out-of-context sanity check: should answer "I don't know."
result = answer_question(qa_chain, "What is the capital of France?")
print(result["answer"])

# Step 7: Chat loop\n\nRun the cell below and type questions at the prompt. Type `exit` to stop.

In [ ]:
while True:
    question = input("You: ")
    if question.strip().lower() == "exit":
        print("Goodbye!")
        break

    result = answer_question(qa_chain, question)
    print("Bot:", result["answer"])
    print("Sources:", result["sources"])